<a href="https://colab.research.google.com/github/kiriakosgp/papadopoulos_av_analysis/blob/main/late_fusion_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

modalities = ["text", "audio", "image"]
predictions_files = {
    "text": "/content/drive/MyDrive/final_text_predictions.json",
    "audio": "/content/drive/MyDrive/audio_predictions.json",
    "image": "/content/drive/MyDrive/finalimage.json"
}

modality_preds = {}

for m in modalities:
    with open(predictions_files[m], "r") as f:
        data = json.load(f)

        modality_preds[m] = {str(d["video_id"]): (int(d["predicted_label"]), float(d["confidence"])) for d in data}

all_video_ids = sorted(set().union(*[modality_preds[m].keys() for m in modalities]))
records = []

for vid in all_video_ids:
    entry = {"video_id": vid}
    for m in modalities:
        pred, conf = modality_preds[m].get(vid, (None, 0.0))
        entry[f"{m}_pred"] = pred
        entry[f"{m}_conf"] = conf
    records.append(entry)

df = pd.DataFrame(records)

pseudo_labels_file = "/content/drive/MyDrive/multimodal_baselines.json"
with open(pseudo_labels_file, "r") as f:
    pseudo_data = json.load(f)

label_mapping = {"negative": 0, "neutral": 1, "positive": 2}

pseudo_dict = {str(d["video_id"]): label_mapping[d["transcript_pred"]] for d in pseudo_data if d["transcript_pred"] is not None}

for m in modalities:
    y_true, y_pred = [], []
    for idx, row in df.iterrows():
        vid = row["video_id"]
        if vid in pseudo_dict and row[f"{m}_pred"] is not None:
            y_true.append(pseudo_dict[vid])
            y_pred.append(row[f"{m}_pred"])
    print(f"\n=== {m.upper()} MODALITY METRICS ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
    print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))
    print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4, target_names=["negative","neutral","positive"]))

fused_preds_majority = []
fused_preds_conf = []

for idx, row in df.iterrows():
    preds = []
    confs = []
    for m in modalities:
        if row[f"{m}_pred"] is not None:
            preds.append(row[f"{m}_pred"])
            confs.append(row[f"{m}_conf"])
    if len(preds) == 0:
        fused_preds_majority.append(None)
        fused_preds_conf.append(None)
        continue

    # Majority voting
    vote_counts = Counter(preds)
    majority_label = vote_counts.most_common(1)[0][0]
    fused_preds_majority.append(majority_label)

    # Confidence-weighted voting
    weight_dict = defaultdict(float)
    for p, c in zip(preds, confs):
        weight_dict[p] += c
    weighted_label = max(weight_dict, key=weight_dict.get)
    fused_preds_conf.append(weighted_label)

df["fused_majority"] = fused_preds_majority
df["fused_weighted"] = fused_preds_conf

y_true_fused, y_majority, y_weighted = [], [], []

for idx, row in df.iterrows():
    vid = row["video_id"]
    if vid in pseudo_dict:
        y_true_fused.append(pseudo_dict[vid])
        y_majority.append(row["fused_majority"])
        y_weighted.append(row["fused_weighted"])

print("\n=== LATE FUSION METRICS (MAJORITY VOTING) ===")
print("Accuracy:", accuracy_score(y_true_fused, y_majority))
print("Balanced Accuracy:", balanced_accuracy_score(y_true_fused, y_majority))
print("Macro F1:", f1_score(y_true_fused, y_majority, average="macro"))
print("Weighted F1:", f1_score(y_true_fused, y_majority, average="weighted"))

print("\n=== LATE FUSION METRICS (CONFIDENCE-WEIGHTED) ===")
print("Accuracy:", accuracy_score(y_true_fused, y_weighted))
print("Balanced Accuracy:", balanced_accuracy_score(y_true_fused, y_weighted))
print("Macro F1:", f1_score(y_true_fused, y_weighted, average="macro"))
print("Weighted F1:", f1_score(y_true_fused, y_weighted, average="weighted"))

cm_majority = confusion_matrix(y_true_fused, y_majority)
cm_weighted = confusion_matrix(y_true_fused, y_weighted)

plt.figure(figsize=(6,5))
sns.heatmap(cm_majority, annot=True, fmt="d", xticklabels=["negative","neutral","positive"], yticklabels=["negative","neutral","positive"], cmap="Blues")
plt.title("Late Fusion Confusion Matrix (Majority) (παραλλαγή)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

plt.figure(figsize=(6,5))
sns.heatmap(cm_weighted, annot=True, fmt="d", xticklabels=["negative","neutral","positive"], yticklabels=["negative","neutral","positive"], cmap="Blues")
plt.title("Late Fusion Confusion Matrix (Weighted Voting) (παραλλαγή)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

fused_results = []
for idx, row in df.iterrows():
    fused_results.append({
        "video_id": row["video_id"],
        "fused_majority": row["fused_majority"],
        "fused_weighted": row["fused_weighted"]
    })

json_path = "/content/drive/MyDrive/results/late_fusion_text_newsplits/fused_predictions.json"
csv_path  = "/content/drive/MyDrive/results/late_fusion_text_newsplits/fused_predictions.csv"

with open(json_path, "w") as f:
    json.dump(fused_results, f, indent=2)

df_fused = pd.DataFrame(fused_results)
df_fused.to_csv(csv_path, index=False)

print(f"Saved fused predictions to {json_path} and {csv_path}")